In [2]:
import pandas as pd

df = pd.read_excel("DF_FINAL.xlsx")

df.head()

calles_base = [
        "Avenida Arcentales",
        "Avenida Menéndez Pelayo",
        "Avenida de la Albufera",
        "Calle Alcalá",
        "Calle Camino de Vinateros",
        "Calle Doctor Esquerdo",
        "Calle José Ortega y Gasset",
        "Calle San Cipriano"
]

calles_este = []

for base in calles_base:
    for col in df.columns:
        if base in col:
            calles_este.append(col)

L = 3  # número de lags

X = pd.DataFrame()

for street in calles_este:
    for lag in range(1, L + 1):
        X[f"{street}_t-{lag}"] = df[street].shift(lag)

# Variables externas reales
X["AEMET_tmed"] = df["AEMET_tmed"]
X["AEMET_prec"] = df["AEMET_prec"]
X["dia_semana"] = df["dia_semana"]
X["festivo"] = df["festivo"]

# Convertir variables categóricas a numéricas
X["dia_semana"] = X["dia_semana"].map({
    "lunes": 0,
    "martes": 1,
    "miércoles": 2,
    "jueves": 3,
    "viernes": 4,
    "sábado": 5,
    "domingo": 6
})

X["festivo"] = X["festivo"].map({
    "no": 0,
    "sí": 1
})


y_col = [col for col in df.columns if "Calle Alcalá" in col][0]
y = df[y_col]

# Limpiar NaNs
X = X.dropna()
y = y.loc[X.index]



In [3]:
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]

y_train = y.iloc[:split]
y_test  = y.iloc[split:]

In [4]:
X_train

,Avenida Arcentales_E-O_t-1,Avenida Arcentales_E-O_t-2,Avenida Arcentales_E-O_t-3,Avenida Arcentales_O-E_t-1,Avenida Arcentales_O-E_t-2,Avenida Arcentales_O-E_t-3,Avenida Menéndez Pelayo_N-S_t-1,Avenida Menéndez Pelayo_N-S_t-2,Avenida Menéndez Pelayo_N-S_t-3,Avenida Menéndez Pelayo_S-N_t-1,...,Calle San Cipriano_E-O_t-1,Calle San Cipriano_E-O_t-2,Calle San Cipriano_E-O_t-3,Calle San Cipriano_O-E_t-1,Calle San Cipriano_O-E_t-2,Calle San Cipriano_O-E_t-3,AEMET_tmed,AEMET_prec,dia_semana,festivo
3,12.0,23.0,35.0,24.0,30.0,31.0,80.0,89.0,217.0,51.0,...,12.0,20.0,16.0,20.0,19.0,44.0,19.6,0.0,4.0,0.0
4,20.0,12.0,23.0,23.0,24.0,30.0,52.0,80.0,89.0,37.0,...,22.0,12.0,20.0,14.0,20.0,19.0,19.6,0.0,4.0,0.0
5,46.0,20.0,12.0,69.0,23.0,24.0,57.0,52.0,80.0,45.0,...,50.0,22.0,12.0,40.0,14.0,20.0,19.6,0.0,4.0,0.0
6,122.0,46.0,20.0,299.0,69.0,23.0,86.0,57.0,52.0,216.0,...,127.0,50.0,22.0,121.0,40.0,14.0,19.6,0.0,4.0,0.0
7,417.0,122.0,46.0,755.0,299.0,69.0,261.0,86.0,57.0,1177.0,...,189.0,127.0,50.0,177.0,121.0,40.0,19.6,0.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25046,392.0,241.0,246.0,180.0,159.0,179.0,565.0,496.0,456.0,317.0,...,151.0,156.0,148.0,236.0,226.0,253.0,32.2,0.0,4.0,0.0
25047,366.0,392.0,241.0,108.0,180.0,159.0,479.0,565.0,496.0,253.0,...,129.0,151.0,156.0,205.0,236.0,226.0,32.2,0.0,4.0,0.0
25048,254.0,366.0,392.0,103.0,108.0,180.0,328.0,479.0,565.0,219.0,...,122.0,129.0,151.0,162.0,205.0,236.0,32.2,0.0,4.0,0.0
25049,242.0,254.0,366.0,113.0,103.0,108.0,327.0,328.0,479.0,246.0,...,162.0,122.0,129.0,193.0,162.0,205.0,32.2,0.0,4.0,0.0


In [5]:
with pd.ExcelWriter("dataset_RF_Alcala.xlsx") as writer:
    X_train.to_excel(writer, sheet_name="X_train")
    X_test.to_excel(writer, sheet_name="X_test")
    y_train.to_excel(writer, sheet_name="y_train")
    y_test.to_excel(writer, sheet_name="y_test")

In [6]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=20,
    random_state=0,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_leaf=20, n_estimators=400,
                      n_jobs=-1, random_state=0)

In [16]:
# Predicción en test
y_pred = rf.predict(X_test)

# Predicción próxima hora
X_next = X.iloc[[-1]]
y_next = rf.predict(X_next)

comparacion = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

comparacion.to_excel("comparacion_predicciones2.xlsx", index=True)

print("Predicción tráfico en Calle Alcalá (E-O) próxima hora:", y_next[0])

Predicción tráfico en Calle Alcalá (E-O) próxima hora: 998.0431322835315


In [8]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Cálculo de métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

def mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100

mape_val = mape(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape_val:.2f} %")

MAE: 74.67
RMSE: 127.33
MAPE: 9.69 %


In [9]:
importances = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(10))

Calle Alcalá_E-O_t-1                  0.712481
Calle Camino de Vinateros_E-O_t-1     0.150751
Avenida Menéndez Pelayo_S-N_t-1       0.088927
Calle Alcalá_O-E_t-1                  0.003585
Calle Alcalá_E-O_t-2                  0.003339
dia_semana                            0.003305
Avenida de la Albufera_E-O_t-3        0.002884
Avenida de la Albufera_O-E_t-1        0.002734
Calle José Ortega y Gasset_E-O_t-3    0.002538
Calle Alcalá_E-O_t-3                  0.001938
dtype: float64


In [13]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

perm_df.head(10)


,feature,importance
18,Calle Alcalá_E-O_t-1,199.601096
24,Calle Camino de Vinateros_E-O_t-1,94.451957
15,Avenida de la Albufera_O-E_t-1,15.313836
50,dia_semana,14.939183
9,Avenida Menéndez Pelayo_S-N_t-1,11.587937
19,Calle Alcalá_E-O_t-2,4.933059
14,Avenida de la Albufera_E-O_t-3,4.550354
38,Calle José Ortega y Gasset_E-O_t-3,3.381672
32,Calle Doctor Esquerdo_N-S_t-3,3.295842
20,Calle Alcalá_E-O_t-3,2.756058


In [14]:
top_k = 8
top_features = perm_df.head(top_k)
top_features


,feature,importance
18,Calle Alcalá_E-O_t-1,199.601096
24,Calle Camino de Vinateros_E-O_t-1,94.451957
15,Avenida de la Albufera_O-E_t-1,15.313836
50,dia_semana,14.939183
9,Avenida Menéndez Pelayo_S-N_t-1,11.587937
19,Calle Alcalá_E-O_t-2,4.933059
14,Avenida de la Albufera_E-O_t-3,4.550354
38,Calle José Ortega y Gasset_E-O_t-3,3.381672


In [15]:
top_streets = (
    top_features["feature"]
    .str.split("_")
    .str[0]
    .value_counts()
)

top_streets

feature
Calle Alcalá                  2
Avenida de la Albufera        2
Calle Camino de Vinateros     1
dia                           1
Avenida Menéndez Pelayo       1
Calle José Ortega y Gasset    1
Name: count, dtype: int64